# Notebook 04: Exhaustive & Hybrid Embedding Benchmark
## Evaluasi Komparatif Representasi Teks (Lexical, Static, Contextual, dan Hybrid Fusion) untuk Prediksi Arah USD/IDR

Notebook ini mengimplementasikan pipeline pengujian menyeluruh (exhaustive benchmark) untuk membandingkan metode representasi teks tunggal dan kombinasi multi-embedding (*Hybrid Fusion*) pada korpus judul berita geopolitik (headline). 
Eksperimen ini dirancang untuk menjawab pertanyaan riset utama: **metode representasi teks mana yang paling efektif memberikan sinyal prediktif terhadap pergerakan kurs USD/IDR hari berikutnya ketika diintegrasikan dengan fitur time-series keuangan?**

---

### Taksonomi Representasi yang Diuji

1. **Kelompok 1: Representasi Leksikal / Berbasis Frekuensi (Count-based Sparse):**
   - **Bag-of-Words (BoW)**: Hitungan frekuensi unigram mentah via `CountVectorizer`.
   - **N-Grams**: Pasangan kata majemuk (bi-gram dan tri-gram) untuk menangkap frasa ekonomi.
   - **TF-IDF**: Pembobotan frekuensi kata dengan kebalikan frekuensi dokumen (Baseline Model B Tugas 2).
   - **BM25 (Best Matching 25)**: Perluasan probabilistik TF-IDF dengan saturasi frekuensi non-linear dan normalisasi panjang dokumen.

2. **Kelompok 2: Dense Static Word Embeddings:**
   - **Word2Vec (Skip-Gram)**: Vektor kata padat kontinu yang dilatih pada korpus judul berita (mean-pooling).
   - **FastText**: Vektor kata berbasis subword character n-grams yang tangguh terhadap kata di luar kamus (OOV).
   - **GloVe (Global Vectors)**: Vektor pretrained dari matriks ko-okurensi global korpus Wikipedia (`glove-wiki-gigaword-50`).

3. **Kelompok 3: Contextual General-Purpose Transformers:**
   - **Sentence-Transformers (`all-MiniLM-L6-v2`)**: Arsitektur siamese network 384 dimensi yang dioptimasi khusus untuk representasi semantik kalimat utuh.
   - **RoBERTa (`roberta-base`)**: Transformer bidirectional encoder 768 dimensi dengan dynamic masking yang dilatih pada korpus umum masif.
   - **DeBERTa (`microsoft/deberta-v3-small`)**: Transformer dengan mekanisme disentangled attention yang memisahkan vektor konten dan posisi kata.

4. **Kelompok 4: Contextual Financial Domain-Specific Transformers:**
   - **FinBERT (`ProsusAI/finbert`)**: Model BERT yang di-pretrain khusus pada korpus teks keuangan (Financial PhraseBank, Reuters TRC2, dan laporan tahunan).
   - **FLANG-RoBERTa (`SALT-NLP/FLANG-RoBERTa`)**: Financial Language Model yang dioptimasi untuk terminologi moneter dan makroekonomi.

5. **Kelompok 5: Hybrid Multi-Embedding Fusion (Inovasi Model C Tingkat Lanjut):**
   - **`hybrid_bm25_finbert`**: BM25 (16d) + FinBERT (16d) -> Sinergi kata kunci eksak dan sentimen kalimat.
   - **`hybrid_glove_finbert`**: GloVe (16d) + FinBERT (16d) -> Sinergi semantik statis global dan konteks finansial.
   - **`hybrid_bm25_glove`**: BM25 (16d) + GloVe (16d) -> Sinergi Top-2 metode non-transformer.
   - **`hybrid_dual_finance`**: FinBERT (16d) + FLANG-RoBERTa (16d) -> Gabungan dua transformer domain keuangan terbaik.
   - **`hybrid_tri_level`**: BM25 (11d) + GloVe (10d) + FinBERT (11d) -> Gabungan 3-pilar (Leksikal + Statis + Kontekstual).
   - **`hybrid_all_combined`**: BM25 (7d) + GloVe (6d) + FastText (6d) + FinBERT (7d) + FLANG (6d) -> Gabungan seluruh spektrum representasi.

6. **Baseline Pembanding:**
   - **Model A (Market Only)**: Model murni fitur pasar time-series (tanpa teks) dari Tugas 2.

---

### Metodologi dan Jaminan Bebas Data Leakage

- **Dual-Track Text Cleaning**: Menerapkan pembersihan 6 tahap (Strip HTML, Lowercase, Strip URL, Strip Digit, Strip Punctuation, Whitespace) pada metode leksikal, sekaligus mempertahankan teks alami (cased, punctuation, digit) pada jalur Transformer.
- **Strict Chronological Split**: Train (s/d 31 Desember 2024), Validation (1 Januari 2025 s/d 31 Desember 2025), Test (1 Januari 2026 s/d 31 Agustus 2026).
- **Alignment Konservatif**: Seluruh berita dipetakan ke hari perdagangan pertama setelah publikasi (aturan Tugas 1), sehingga fitur teks hari t hanya memanfaatkan berita sebelum hari t.
- **Harmonisasi Dimensi Tetap 32 Fitur Teks**: Baik metode tunggal maupun hybrid diproyeksikan ke total 32 komponen via PCA/TruncatedSVD (fit HANYA pada Train split).
- **Multi-Classifier Benchmarking**: Menguji LightGBM, Random Forest, dan Logistic Regression (L2) untuk menemukan arsitektur terbaik bagi masing-masing representasi.

In [ ]:
import os
import sys
import time
import json
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# Scikit-learn
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss, matthews_corrcoef
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# PyTorch & Device Detection
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Suppress unnecessary socket warnings & progress bars
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Direktori proyek
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebook' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
ALIGNED_DIR = DATA_DIR / 'aligned'
CLEANED_DIR = DATA_DIR / 'cleaned'
CACHE_DIR = PROCESSED_DIR / 'embeddings_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TARGET_COL = 'target_next_up'
MARKET_FEATURES = ['ret_1d', 'ret_2d', 'ret_3d', 'ret_5d', 'rate_to_ma5', 'rate_to_ma20', 'vol_5', 'vol_20']

print(f"[Setup] PyTorch version: {torch.__version__}")
print(f"[Setup] Komputasi device: {device}")
if device.type == 'cuda':
    print(f"[Setup] GPU Name: {torch.cuda.get_device_name(0)}")
print(f"[Setup] Direktori cache: {CACHE_DIR.resolve()}")

In [ ]:
# 1. Memuat dataset harian yang sudah dipisahkan (Train, Validation, Test)
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv', parse_dates=['trading_date'])
val_df = pd.read_csv(PROCESSED_DIR / 'validation.csv', parse_dates=['trading_date'])
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv', parse_dates=['trading_date'])

# Menggabungkan seluruh rentang hari perdagangan untuk mapping konsisten
full_market_df = pd.concat([train_df, val_df, test_df], ignore_index=True).sort_values('trading_date').reset_index(drop=True)
all_trading_dates = full_market_df['trading_date'].dt.strftime('%Y-%m-%d').values

# Index masker per split
train_mask = (full_market_df['trading_date'] <= '2024-12-31').values
val_mask = ((full_market_df['trading_date'] >= '2025-01-01') & (full_market_df['trading_date'] <= '2025-12-31')).values
test_mask = (full_market_df['trading_date'] >= '2026-01-01').values

# 2. Memuat berita geopolitik dan pemetaannya ke tanggal perdagangan
news_df = pd.read_csv(CLEANED_DIR / 'news_cleaned.csv', usecols=['article_id', 'title_clean'])
align_df = pd.read_csv(ALIGNED_DIR / 'news_alignment.csv', usecols=['article_id', 'effective_trading_date'])
align_df['effective_trading_date'] = pd.to_datetime(align_df['effective_trading_date']).dt.strftime('%Y-%m-%d')

merged_news = align_df.merge(news_df, on='article_id', how='inner')
merged_news['title_clean'] = merged_news['title_clean'].fillna('')

print(f"[Data] Train samples: {train_df.shape[0]} hari ({train_df['trading_date'].min().date()} s/d {train_df['trading_date'].max().date()})")
print(f"[Data] Val samples:   {val_df.shape[0]} hari ({val_df['trading_date'].min().date()} s/d {val_df['trading_date'].max().date()})")
print(f"[Data] Test samples:  {test_df.shape[0]} hari ({test_df['trading_date'].min().date()} s/d {test_df['trading_date'].max().date()})")
print(f"[Data] Total artikel berita terpetakan: {merged_news.shape[0]} baris")
print(f"[Data] Cakupan hari perdagangan: {merged_news['effective_trading_date'].nunique()} dari {len(all_trading_dates)} hari")

### Dual-Track Text Preprocessing Pipeline

Pada tahap ini, kita menerapkan strategi pembersihan teks yang terbagi menjadi dua jalur:

1. **Jalur Leksikal & Static (BoW, N-Grams, TF-IDF, BM25, Word2Vec, FastText):**
   Menerapkan pipeline pembersihan 6 tahap lengkap:
   - **Tahap 1 (Strip HTML):** Menghapus tag HTML seperti `<a>`, `<div>`, dll.
   - **Tahap 2 (Lowercase):** Mengubah seluruh teks menjadi huruf kecil agar pencocokan kata konsisten.
   - **Tahap 3 (Strip URL):** Menghapus pola link website (`http://`, `https://`, `www.`).
   - **Tahap 4 (Strip Digit):** Menghapus karakter angka numerik (`0-9`) untuk menghindari ledakan kosakata acak.
   - **Tahap 5 (Strip Punctuation):** Menghapus simbol tanda baca (`:`, `.`, `;`, `+`, `%`, `!`, dll.).
   - **Tahap 6 (Final Clean Text):** Menormalkan spasi ganda dan spasi tepi (*whitespace trimming*).

2. **Jalur Transformer Kontekstual (FinBERT, FLANG-RoBERTa, RoBERTa, DeBERTa, MiniLM):**
   Mempertahankan huruf kapital, tanda baca, serta angka numerik (seperti `+4.5%`, `$591 million`, `U.S. Navy`, `Fed`). Tokenizer subword modern (*Byte-Pair Encoding* dan *WordPiece*) secara inheren dilatih pada teks alami dan membutuhkan kapitalisasi untuk membedakan entitas nama negara/pejabat serta angka untuk menangkap intensitas sentimen pasar.

In [ ]:
def clean_text_lexical_pipeline(text: str) -> str:
    """
    Pembersihan teks 6 tahap untuk representasi leksikal dan vektor statis:
    0. Raw Input
    1. Strip HTML tags
    2. Lowercase
    3. Strip URL
    4. Strip Digit
    5. Strip Punctuation
    6. Final Clean Text (Normalisasi spasi)
    """
    if not isinstance(text, str):
        return ""
    # 1. Strip HTML
    t = re.sub(r'<[^>]+>', ' ', text)
    # 2. Lowercase
    t = t.lower()
    # 3. Strip URL
    t = re.sub(r"https?://\S+|www\.\S+", ' ', t)
    # 4. Strip Digit
    t = re.sub(r"\d+", ' ', t)
    # 5. Strip Punctuation
    t = re.sub(r"[^\w\s]", ' ', t)
    # 6. Final Clean Text
    t = re.sub(r"\s+", ' ', t).strip()
    return t

# Verifikasi fungsi 6 tahap dengan contoh kalimat uji
sample_raw = 'BREAKING: <a href="https://www.cnbc.com/oil-crisis">U.S. Navy</a> Strikes 3 Iranian Oil Tankers After Missile Attacks; Crude Jumps +4.5% (2026)!'
print("[Verifikasi Pembersihan 6 Tahap]:")
print("Input Mentah  :", sample_raw)
print("Output Bersih :", clean_text_lexical_pipeline(sample_raw))

# Menerapkan pada dataset artikel
print("\nMenerapkan pembersihan teks pada 14.941 artikel...")
merged_news['title_lexical'] = merged_news['title_clean'].map(clean_text_lexical_pipeline)
merged_news['title_transformer'] = merged_news['title_clean']
print("[Selesai] Teks leksikal (title_lexical) dan teks alami transformer (title_transformer) siap digunakan.")

In [ ]:
# Implementasi mandiri BM25Vectorizer berbasis formulasi BM25Okapi
class BM25Vectorizer:
    """
    Vektorizer representasi dokumen berbasis BM25Okapi standar.
    Menggunakan parameter default: k1=1.5 dan b=0.75.
    """
    def __init__(self, k1=1.5, b=0.75, max_features=5000):
        self.k1 = k1
        self.b = b
        self.max_features = max_features
        self.cv = CountVectorizer(max_features=max_features)
        self.idf_ = None
        self.avgdl_ = None

    def fit(self, raw_documents):
        X = self.cv.fit_transform(raw_documents)
        N = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1])
        # Formula IDF BM25: ln((N - df + 0.5) / (df + 0.5) + 1.0)
        self.idf_ = np.log((N - df + 0.5) / (df + 0.5) + 1.0)
        doc_len = np.array(X.sum(axis=1)).flatten()
        self.avgdl_ = np.mean(doc_len) if np.mean(doc_len) > 0 else 1.0
        return self

    def transform(self, raw_documents):
        from scipy.sparse import csr_matrix
        X = self.cv.transform(raw_documents)
        doc_len = np.array(X.sum(axis=1)).flatten()
        rows, cols = X.nonzero()
        data = X.data
        denom = data + self.k1 * (1.0 - self.b + self.b * (doc_len[rows] / self.avgdl_))
        num = self.idf_[cols] * data * (self.k1 + 1.0)
        bm25_data = num / (denom + 1e-9)
        return csr_matrix((bm25_data, (rows, cols)), shape=X.shape)

    def fit_transform(self, raw_documents):
        return self.fit(raw_documents).transform(raw_documents)

# Membuat korpus dokumen harian (daily text) menggunakan teks yang sudah dibersihkan secara leksikal
daily_lexical_series = merged_news.groupby('effective_trading_date')['title_lexical'].apply(lambda titles: ' '.join(titles))
daily_lexical_texts = [daily_lexical_series.get(d, '') for d in all_trading_dates]

def extract_lexical_representations(texts, train_mask, n_components=32):
    """
    Mengekstrak representasi leksikal: BoW, N-Grams, TF-IDF, dan BM25.
    TruncatedSVD di-fit secara ketat HANYA pada data Train untuk mencegah kebocoran data.
    """
    train_texts = [texts[i] for i, m in enumerate(train_mask) if m]
    lexical_results = {}
    
    extractors = {
        'bow_svd': CountVectorizer(max_features=5000),
        'ngrams_svd': CountVectorizer(ngram_range=(2, 3), max_features=5000),
        'tfidf_svd': TfidfVectorizer(max_features=5000, sublinear_tf=True),
        'bm25_svd': BM25Vectorizer(max_features=5000)
    }
    
    for name, extractor in extractors.items():
        print(f"[Lexical] Memproses {name}...")
        extractor.fit(train_texts)
        X_sparse = extractor.transform(texts)
        
        svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_SEED)
        svd.fit(X_sparse[train_mask])
        X_dense = svd.transform(X_sparse)
        lexical_results[name] = X_dense
        
    return lexical_results

lexical_embeddings = extract_lexical_representations(daily_lexical_texts, train_mask, n_components=32)
print("[Lexical] Seluruh representasi leksikal selesai diekstrak.")

In [ ]:
from gensim.models import Word2Vec, FastText
import gensim.downloader as api

def get_static_embeddings(merged_news, all_trading_dates, train_mask):
    """
    Mengekstrak representasi vektor statis: Word2Vec, FastText, dan GloVe.
    Word2Vec dan FastText dilatih pada token judul leksikal split Train (bebas leakage).
    Representasi harian dibentuk via mean pooling artikel per hari perdagangan.
    """
    static_results = {}
    
    tokenized_titles = [t.split() for t in merged_news['title_lexical']]
    train_dates_set = set(all_trading_dates[train_mask])
    train_tokenized = [
        t.split() 
        for date, t in zip(merged_news['effective_trading_date'], merged_news['title_lexical'])
        if date in train_dates_set
    ]
    
    # --- Word2Vec ---
    w2v_path = CACHE_DIR / 'w2v_daily.npy'
    if w2v_path.exists():
        print("[Static] Memuat Word2Vec dari cache...")
        static_results['word2vec'] = np.load(w2v_path)
    else:
        print("[Static] Melatih Word2Vec pada korpus train...")
        w2v_model = Word2Vec(sentences=train_tokenized, vector_size=100, window=5, min_count=2, seed=RANDOM_SEED, workers=4)
        article_vecs = []
        for tokens in tokenized_titles:
            valid_vecs = [w2v_model.wv[tok] for tok in tokens if tok in w2v_model.wv]
            article_vecs.append(np.mean(valid_vecs, axis=0) if len(valid_vecs) > 0 else np.zeros(100))
        merged_news['w2v_vec'] = list(article_vecs)
        daily_w2v_map = merged_news.groupby('effective_trading_date')['w2v_vec'].apply(lambda vecs: np.mean(np.vstack(vecs), axis=0)).to_dict()
        daily_w2v = np.array([daily_w2v_map.get(d, np.zeros(100)) for d in all_trading_dates])
        np.save(w2v_path, daily_w2v)
        static_results['word2vec'] = daily_w2v

    # --- FastText ---
    ft_path = CACHE_DIR / 'fasttext_daily.npy'
    if ft_path.exists():
        print("[Static] Memuat FastText dari cache...")
        static_results['fasttext'] = np.load(ft_path)
    else:
        print("[Static] Melatih FastText pada korpus train...")
        ft_model = FastText(sentences=train_tokenized, vector_size=100, window=5, min_count=2, seed=RANDOM_SEED, workers=4)
        article_vecs = []
        for tokens in tokenized_titles:
            vecs = [ft_model.wv[tok] for tok in tokens] if len(tokens) > 0 else []
            article_vecs.append(np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(100))
        merged_news['ft_vec'] = list(article_vecs)
        daily_ft_map = merged_news.groupby('effective_trading_date')['ft_vec'].apply(lambda vecs: np.mean(np.vstack(vecs), axis=0)).to_dict()
        daily_ft = np.array([daily_ft_map.get(d, np.zeros(100)) for d in all_trading_dates])
        np.save(ft_path, daily_ft)
        static_results['fasttext'] = daily_ft

    # --- GloVe ---
    glove_path = CACHE_DIR / 'glove_daily.npy'
    if glove_path.exists():
        print("[Static] Memuat GloVe dari cache...")
        static_results['glove'] = np.load(glove_path)
    else:
        print("[Static] Memuat GloVe pretrained (glove-wiki-gigaword-50)...")
        try:
            glove_model = api.load('glove-wiki-gigaword-50')
            article_vecs = []
            for tokens in tokenized_titles:
                valid_vecs = [glove_model[tok] for tok in tokens if tok in glove_model]
                article_vecs.append(np.mean(valid_vecs, axis=0) if len(valid_vecs) > 0 else np.zeros(50))
            merged_news['glove_vec'] = list(article_vecs)
            daily_glove_map = merged_news.groupby('effective_trading_date')['glove_vec'].apply(lambda vecs: np.mean(np.vstack(vecs), axis=0)).to_dict()
            daily_glove = np.array([daily_glove_map.get(d, np.zeros(50)) for d in all_trading_dates])
            np.save(glove_path, daily_glove)
            static_results['glove'] = daily_glove
        except Exception as e:
            print(f"[Warning] Gagal memuat GloVe via downloader: {e}. GloVe dilewati.")

    return static_results

static_embeddings = get_static_embeddings(merged_news, all_trading_dates, train_mask)
print("[Static] Seluruh representasi statis selesai diproses.")

In [ ]:
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

def extract_transformer_headline_embeddings(model_id, headlines, batch_size=128):
    """
    Ekstraksi vektor embedding dari Transformer dengan GPU batching dan mean pooling token.
    Menggunakan teks alami (title_transformer) yang mempertahankan kapitalisasi dan simbol finansial.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(device)
    model.eval()
    
    all_embeddings = []
    total = len(headlines)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            batch_texts = headlines[i:i + batch_size]
            encoded = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=96, 
                return_tensors='pt'
            ).to(device)
            
            outputs = model(**encoded)
            token_embeddings = outputs.last_hidden_state
            attention_mask = encoded['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
            
            sum_embeddings = torch.sum(token_embeddings * attention_mask, dim=1)
            sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_pooled = (sum_embeddings / sum_mask).cpu().numpy()
            all_embeddings.append(mean_pooled)
            
    del model, tokenizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
            
    return np.vstack(all_embeddings)

def get_contextual_embeddings(merged_news, all_trading_dates):
    """
    Ekstraksi seluruh model transformer kontekstual dengan caching per model.
    """
    transformer_models = {
        'all_minilm_l6': 'sentence-transformers/all-MiniLM-L6-v2',
        'roberta_base': 'roberta-base',
        'deberta_v3_small': 'microsoft/deberta-v3-small',
        'finbert_prosus': 'ProsusAI/finbert',
        'flang_roberta': 'SALT-NLP/FLANG-RoBERTa'
    }
    
    headlines_list = merged_news['title_transformer'].tolist()
    contextual_results = {}
    
    for key, model_id in transformer_models.items():
        cache_file = CACHE_DIR / f"{key}_daily.npy"
        if cache_file.exists():
            print(f"[Transformer] Memuat {key} dari cache...")
            contextual_results[key] = np.load(cache_file)
            continue
            
        print(f"[Transformer] Mengekstrak {key} ({model_id}) pada device: {device}...")
        t_start = time.time()
        
        try:
            if key == 'all_minilm_l6':
                st_model = SentenceTransformer(model_id, device=str(device))
                article_embs = st_model.encode(headlines_list, batch_size=128, show_progress_bar=False)
            else:
                article_embs = extract_transformer_headline_embeddings(model_id, headlines_list, batch_size=128)
                
            merged_news[f'{key}_vec'] = list(article_embs)
            daily_map = merged_news.groupby('effective_trading_date')[f'{key}_vec'].apply(
                lambda vecs: np.mean(np.vstack(vecs), axis=0)
            ).to_dict()
            
            dim_size = article_embs.shape[1]
            daily_vecs = np.array([daily_map.get(d, np.zeros(dim_size)) for d in all_trading_dates])
            
            np.save(cache_file, daily_vecs)
            contextual_results[key] = daily_vecs
            print(f"[Transformer] {key} selesai dalam {time.time() - t_start:.2f} detik (dimensi: {dim_size})")
        except Exception as e:
            print(f"[Error] Gagal mengekstrak {key}: {e}. Metode dilewati.")
            
    return contextual_results

contextual_embeddings = get_contextual_embeddings(merged_news, all_trading_dates)
print("[Transformer] Seluruh representasi transformer kontekstual selesai diproses.")

### Konstruksi Representasi Tunggal & Hybrid Multi-Embedding (32 Dimensi)

Untuk menjamin evaluasi yang adil (*ceteris paribus*), seluruh representasi (baik metode tunggal maupun gabungan hybrid) distandarisasi ke **total 32 dimensi fitur teks** sebelum digabungkan dengan 8 fitur pasar teknikal:

1. **Representasi Tunggal (Single):** Leksikal (BoW, N-Grams, TF-IDF, BM25), Statis (Word2Vec, FastText, GloVe), dan Kontekstual (MiniLM, RoBERTa, DeBERTa, FinBERT, FLANG).
2. **Representasi Hybrid (Multi-Embedding Fusion):**
   - **`hybrid_bm25_finbert`**: BM25 (16d) + FinBERT (16d). Sinergi kata kunci eksak dan pemahaman sentimen kalimat.
   - **`hybrid_glove_finbert`**: GloVe (16d) + FinBERT (16d). Sinergi kemiripan topik statis global dan konteks finansial (mirip M4a teman).
   - **`hybrid_bm25_glove`**: BM25 (16d) + GloVe (16d). Kombinasi dua metode terbaik dari leaderboard tunggal.
   - **`hybrid_dual_finance`**: FinBERT (16d) + FLANG-RoBERTa (16d). Gabungan dua transformer spesifik domain finansial.
   - **`hybrid_tri_level`**: BM25 (11d) + GloVe (10d) + FinBERT (11d). Sinergi 3 pilar: Leksikal + Statis + Kontekstual.
   - **`hybrid_all_combined`**: BM25 (7d) + GloVe (6d) + FastText (6d) + FinBERT (7d) + FLANG (6d). Seluruh spektrum representasi (mirip M4b teman).

In [ ]:
# 1. Menyiapkan Fitur Pasar Murni (Model A) dan Target
X_market = full_market_df[MARKET_FEATURES].values
y = full_market_df[TARGET_COL].values

def reduce_pca_train_only(matrix, n_comp):
    """
    Standarisasi dan reduksi dimensi PCA yang di-fit HANYA pada data Train (bebas data leakage).
    """
    scaler = StandardScaler()
    pca = PCA(n_components=n_comp, random_state=RANDOM_SEED)
    scaled_train = scaler.fit_transform(matrix[train_mask])
    pca.fit(scaled_train)
    return pca.transform(scaler.transform(matrix))

# Koleksi seluruh representasi teks (Single & Hybrid)
all_representations = {}

# --- A. Single Representations (32 Dimensi) ---
for name, mat in lexical_embeddings.items():
    all_representations[name] = ('Count / Lexical', mat)

for name in ['word2vec', 'fasttext', 'glove']:
    if name in static_embeddings:
        mat = static_embeddings[name]
        mat_reduced = reduce_pca_train_only(mat, min(32, mat.shape[1]))
        all_representations[name] = ('Dense Static Vectors', mat_reduced)

for name, mat in contextual_embeddings.items():
    mat_reduced = reduce_pca_train_only(mat, 32)
    all_representations[name] = ('Contextual Transformer', mat_reduced)

# --- B. Hybrid Multi-Embedding Representations (Total 32 Dimensi Teks) ---
# 1. Lexical + FinBERT (BM25 16d + FinBERT 16d)
if 'bm25_svd' in lexical_embeddings and 'finbert_prosus' in contextual_embeddings:
    all_representations['hybrid_bm25_finbert'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(lexical_embeddings['bm25_svd'], 16),
            reduce_pca_train_only(contextual_embeddings['finbert_prosus'], 16)
        ])
    )

# 2. GloVe + FinBERT (GloVe 16d + FinBERT 16d - Mirip M4a)
if 'glove' in static_embeddings and 'finbert_prosus' in contextual_embeddings:
    all_representations['hybrid_glove_finbert'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(static_embeddings['glove'], 16),
            reduce_pca_train_only(contextual_embeddings['finbert_prosus'], 16)
        ])
    )

# 3. Top-2 Leaderboard Standalone (BM25 16d + GloVe 16d)
if 'bm25_svd' in lexical_embeddings and 'glove' in static_embeddings:
    all_representations['hybrid_bm25_glove'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(lexical_embeddings['bm25_svd'], 16),
            reduce_pca_train_only(static_embeddings['glove'], 16)
        ])
    )

# 4. Dual Financial Domain (FinBERT 16d + FLANG-RoBERTa 16d)
if 'finbert_prosus' in contextual_embeddings and 'flang_roberta' in contextual_embeddings:
    all_representations['hybrid_dual_finance'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(contextual_embeddings['finbert_prosus'], 16),
            reduce_pca_train_only(contextual_embeddings['flang_roberta'], 16)
        ])
    )

# 5. Tri-Hybrid 3-Pilar (BM25 11d + GloVe 10d + FinBERT 11d = 32d)
if 'bm25_svd' in lexical_embeddings and 'glove' in static_embeddings and 'finbert_prosus' in contextual_embeddings:
    all_representations['hybrid_tri_level'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(lexical_embeddings['bm25_svd'], 11),
            reduce_pca_train_only(static_embeddings['glove'], 10),
            reduce_pca_train_only(contextual_embeddings['finbert_prosus'], 11)
        ])
    )

# 6. All-Combined Hybrid (BM25 7d + GloVe 6d + FastText 6d + FinBERT 7d + FLANG 6d = 32d - Mirip M4b)
if all(k in lexical_embeddings or k in static_embeddings or k in contextual_embeddings 
       for k in ['bm25_svd', 'glove', 'fasttext', 'finbert_prosus', 'flang_roberta']):
    all_representations['hybrid_all_combined'] = (
        'Hybrid Multi-Embedding',
        np.hstack([
            reduce_pca_train_only(lexical_embeddings['bm25_svd'], 7),
            reduce_pca_train_only(static_embeddings['glove'], 6),
            reduce_pca_train_only(static_embeddings['fasttext'], 6),
            reduce_pca_train_only(contextual_embeddings['finbert_prosus'], 7),
            reduce_pca_train_only(contextual_embeddings['flang_roberta'], 6)
        ])
    )

print(f"[Hybrid] Total representasi yang siap diuji: {len(all_representations)} metode (Single + Hybrid).")

### Evaluasi Terkendali Murni Deep Learning: Multimodal MLP (PyTorch)

Untuk menjamin prinsip keadilan evaluasi paling ketat (*ceteris paribus*), seluruh 18 representasi teks (Single & Hybrid) dievaluasi menggunakan **arsitektur Deep Learning yang seragam (Multi-Layer Perceptron)**:
- **Arsitektur:** Input(40) -> Dense(64) + BatchNorm + ReLU + Dropout(0.3) -> Dense(32) + BatchNorm + ReLU + Dropout(0.3) -> Linear(1)
- **Optimizer:** AdamW (learning rate: 1e-3, weight decay: 1e-4)
- **Loss:** BCEWithLogitsLoss
- **Epochs:** 50 epochs dengan batch size 32
- **Seed:** Terkunci di 42 (deterministik)

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class MultimodalMLP(nn.Module):
    """
    Arsitektur Multi-Layer Perceptron (Deep Learning) standar:
    Input -> Dense(64) -> BatchNorm -> ReLU -> Dropout(0.3)
          -> Dense(32) -> BatchNorm -> ReLU -> Dropout(0.3)
          -> Output Linear(1)
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

def train_eval_mlp(X_tr, y_tr, X_eval, y_eval, epochs=50, batch_size=32, lr=1e-3, seed=RANDOM_SEED):
    """
    Melatih model PyTorch MLP pada data latih dan mengevaluasinya secara deterministik.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_ev_s = scaler.transform(X_eval)
    
    X_tr_t = torch.tensor(X_tr_s, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1).to(device)
    X_ev_t = torch.tensor(X_ev_s, dtype=torch.float32).to(device)
    
    dataset = TensorDataset(X_tr_t, y_tr_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = MultimodalMLP(input_dim=X_tr.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    model.train()
    for epoch in range(epochs):
        for b_x, b_y in loader:
            optimizer.zero_grad()
            loss = criterion(model(b_x), b_y)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        logits = model(X_ev_t)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        preds = (probs >= 0.5).astype(int)
        
    auc = roc_auc_score(y_eval, probs)
    acc = accuracy_score(y_eval, preds)
    f1 = f1_score(y_eval, preds, average='macro')
    mcc = matthews_corrcoef(y_eval, preds)
    brier = brier_score_loss(y_eval, probs)
    
    return {
        'Model': 'MLP (PyTorch)',
        'ROC_AUC': round(auc, 4),
        'Directional_Acc': round(acc, 4),
        'F1_Macro': round(f1, 4),
        'MCC': round(mcc, 4),
        'Brier_Score': round(brier, 4)
    }

print("[MLP] Menjalankan benchmark seragam menggunakan PyTorch MLP...")
# 1. Baseline Model A (Market Only)
market_eval = train_eval_mlp(X_market[train_mask], y[train_mask], X_market[val_mask], y[val_mask])
benchmark_records = [{
    'Method': 'Market Only (Model A Baseline)',
    'Category': 'Time-Series Only',
    'Model': market_eval['Model'],
    'Text_Dimension': 0,
    'Val_ROC_AUC': market_eval['ROC_AUC'],
    'Val_Directional_Acc': market_eval['Directional_Acc'],
    'Val_F1_Macro': market_eval['F1_Macro'],
    'Val_MCC': market_eval['MCC'],
    'Val_Brier_Score': market_eval['Brier_Score']
}]

# 2. Single & Hybrid Embeddings
for name, (cat, text_mat) in all_representations.items():
    X_fused = np.hstack([X_market, text_mat])
    rec = train_eval_mlp(X_fused[train_mask], y[train_mask], X_fused[val_mask], y[val_mask])
    benchmark_records.append({
        'Method': name,
        'Category': cat,
        'Model': rec['Model'],
        'Text_Dimension': text_mat.shape[1],
        'Val_ROC_AUC': rec['ROC_AUC'],
        'Val_Directional_Acc': rec['Directional_Acc'],
        'Val_F1_Macro': rec['F1_Macro'],
        'Val_MCC': rec['MCC'],
        'Val_Brier_Score': rec['Brier_Score']
    })

val_leaderboard = pd.DataFrame(benchmark_records).sort_values(by='Val_ROC_AUC', ascending=False).reset_index(drop=True)
val_leaderboard.index = val_leaderboard.index + 1
val_leaderboard.index.name = 'Rank'

print("=" * 115)
print("LEADERBOARD BENCHMARK VALIDATION SET (TAHUN 2025) - MURNI DEEP LEARNING MLP (PYTORCH)")
print("=" * 115)
print(val_leaderboard.to_string())
print("=" * 115)

output_val_csv = REPORTS_DIR / 'exhaustive_hybrid_validation_benchmark_mlp.csv'
val_leaderboard.to_csv(output_val_csv)
print(f"[Output] Hasil benchmark validasi MLP disimpan ke: {output_val_csv.resolve()}")

### Evaluasi Test Set Final (Tahun 2026) - Murni Deep Learning MLP (PyTorch)

Pengujian satu kali pada data masa depan (Test Set 2026) menggunakan arsitektur PyTorch MLP yang sama persis.

In [ ]:
test_records = []

# Model A (Market Only)
rec_te_m = train_eval_mlp(X_market[train_mask], y[train_mask], X_market[test_mask], y[test_mask])
test_records.append({
    'Representasi': 'M0_Market',
    'Kategori': 'Baseline Pasar',
    'Model': rec_te_m['Model'],
    'Test Accuracy': f"{rec_te_m['Directional_Acc']*100:.2f}%",
    'Test AUC': rec_te_m['ROC_AUC'],
    'Test F1-Macro': rec_te_m['F1_Macro'],
    'Test MCC': f"{rec_te_m['MCC']:+.4f}",
    'Brier Score': rec_te_m['Brier_Score']
})

for name, (cat, text_mat) in all_representations.items():
    X_fused = np.hstack([X_market, text_mat])
    rec_te = train_eval_mlp(X_fused[train_mask], y[train_mask], X_fused[test_mask], y[test_mask])
    test_records.append({
        'Representasi': name,
        'Kategori': cat,
        'Model': rec_te['Model'],
        'Test Accuracy': f"{rec_te['Directional_Acc']*100:.2f}%",
        'Test AUC': rec_te['ROC_AUC'],
        'Test F1-Macro': rec_te['F1_Macro'],
        'Test MCC': f"{rec_te['MCC']:+.4f}",
        'Brier Score': rec_te['Brier_Score']
    })

test_leaderboard = pd.DataFrame(test_records).sort_values(by='Test AUC', ascending=False).reset_index(drop=True)
test_leaderboard.index = test_leaderboard.index + 1
test_leaderboard.index.name = 'Rank'

print("=" * 115)
print("HASIL EVALUASI FINAL TEST SET (TAHUN 2026) - MURNI DEEP LEARNING MLP (PYTORCH)")
print("=" * 115)
print(test_leaderboard.to_string())
print("=" * 115)

output_test_csv = REPORTS_DIR / 'test_set_final_benchmark_mlp.csv'
test_leaderboard.to_csv(output_test_csv)
print(f"[Output] Hasil benchmark test set MLP disimpan ke: {output_test_csv.resolve()}")

### Analisis Temuan dan Implikasi Metodologis (Bahan Laporan Bab 1 dan Bab 3)

Berdasarkan tabel benchmark murni Deep Learning MLP di atas:

1. **Jaminan Keadilan Komparasi (Ceteris Paribus):**
   - Dengan mengunci classifier pada satu arsitektur Deep Learning MLP PyTorch seragam (dengan hyperparameter, optimasi AdamW, dan loss yang sama persis), seluruh variasi performa murni berasal dari kualitas representasi fitur teks (bukan karena algoritma yang berbeda).

2. **Dinamika Representasi Leksikal vs Statis vs Kontekstual vs Hybrid:**
   - Perhatikan posisi metode domain finansial (FinBERT, FLANG) dan varian hybrid ketika diproses oleh neural network non-linear.
   - Lapisan non-linear (ReLU) dan Dropout pada MLP memungkinkan model mempelajari korelasi silang (*cross-feature correlation*) antara pergerakan volatilitas pasar dan nuansa sentimen berita geopolitik.